# Awal

In [28]:
%load_ext autoreload
%autoreload 2
from pathlib import Path
import sys
cwd = Path.cwd().resolve()
REPO_ROOT = next((p for p in [cwd, *cwd.parents] if (p / 'src' / 'rnn' / 'utils' / 'text_utils.py').exists()), cwd)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
# pip install tensorflow
# pip install nltk


In [ ]:
# import numpy as np

# import os
# from preprocessing.image_utils import extract_features

# def extract_and_save_npy(image_dir, model_cnn, output_dir):
        
#     if not os.path.exists(output_dir):
#         os.makedirs(output_dir)
        
#     images_file = [f for f in os.listdir(image_dir) if f.endswith(('.jpg', '.png', '.jpeg'))]

#     features = []
#     names = []

#     for img_name in (images_file):
#         img_path = os.path.join(image_dir, img_name)
#         feat = extract_features(model_cnn, img_path)

#         features.append(feat)
#         names.append(img_name)

#     features_matrix = np.array(features)
#     names_array = np.array(names)

#     np.save(os.path.join(output_dir, "features.npy"), features_matrix)
#     np.save(os.path.join(output_dir, "image_names.npy"), names_array)

#     print("Selesai! Fitur dan Nama Gambar telah disimpan.")


In [29]:
import tensorflow
import os
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.models import Model
import numpy as np

image_dir = "../../dataset/Images"
output_dir = "../../images_feature"

base_model = InceptionV3(weights='imagenet')
model_cnn = Model(inputs=base_model.input, outputs=base_model.get_layer('avg_pool').output)

features_path = os.path.join(output_dir, "features.npy")
names_path = os.path.join(output_dir, "image_names.npy")

# if not os.path.exists(features_path) or not os.path.exists(names_path):
#     print("Memulai proses ekstraksi...")
#     extract_and_save_npy(image_dir, model_cnn, output_dir)
# else:
#     print("Load Feature...")


features_matrix = np.load(features_path)
image_names = np.load(names_path)

image_features_map = dict(zip(image_names, features_matrix))


#### Captions

In [30]:
from src.rnn.utils.text_utils import CaptionPreprocessor

text_util = CaptionPreprocessor()
text_util.build_vocabulary(str(REPO_ROOT / "RNN_dataset" / "captions.txt"))
map_image_cap = text_util.get_image_to_captions_mapping(str(REPO_ROOT / "RNN_dataset" / "captions.txt"))
data = {}

for image in map_image_cap:
    captions = map_image_cap[image]
    seq = text_util.texts_to_sequences(captions=captions)
    padded_seq = text_util.pad_sequences(sequences=seq)
    data[image] = padded_seq




### Bagian 3

In [31]:
import tensorflow as tf
import time
import pandas as pd
from src.rnn.utils.train_utils import DataGenerator
from tensorflow.keras.layers import Input, Dense, Embedding, LSTM, SimpleRNN, Concatenate, Reshape
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Masking

def build_model(lstm: bool, layers: int, hidden_state_count: int, vocab_size: int, sequence_length: int = 35):
    image_input = Input(shape=(2048,), name="Image_Input")
    caption_input = Input(shape=(sequence_length,), name="Caption_Input")

    image_projection = Dense(256, activation='relu')(image_input)
    image_projection = Reshape((1, 256))(image_projection)
    image_projection = Masking(mask_value=0.0)(image_projection) 

    caption_embedding = Embedding(input_dim=vocab_size, output_dim=256, mask_zero=True)(caption_input)

    merged = Concatenate(axis=1)([image_projection, caption_embedding])

    x = merged
    for i in range(layers):
        if lstm:
            x = LSTM(hidden_state_count, return_sequences=True, name=f"LSTM_Layer_{i+1}")(x)
        else:
            x = SimpleRNN(hidden_state_count, return_sequences=True, name=f"RNN_Layer_{i+1}")(x)

    x = tf.keras.layers.Lambda(lambda t: t[:, 1:, :])(x)
    
    outputs = Dense(vocab_size, activation='softmax', name="Output_Layer")(x)

    model = Model(inputs=[image_input, caption_input], outputs=outputs)
    model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

    model.summary()
    return model


In [32]:
variations = [
    {"layers": 1, "hidden": 128, "name": "Shallow_Small"},
    {"layers": 2, "hidden": 128, "name": "Deep_Small"},
    {"layers": 3, "hidden": 128, "name": "VeryDeep_Small"},
    {"layers": 1, "hidden": 256, "name": "Shallow_Mid"},
    {"layers": 1, "hidden": 512, "name": "Shallow_Large"},
    {"layers": 2, "hidden": 512, "name": "Deep_Large"},
]

# Tentukan parameter umum
VOCAB_SIZE = text_util.vocab_size
SEQ_LEN = text_util.sequence_length
EPOCHS = 5
BATCH_SIZE = 64


In [33]:
all_images = list(data.keys())
np.random.seed(42)
np.random.shuffle(all_images)

train_keys = all_images[:6000]
val_keys   = all_images[6000:7000]
test_keys  = all_images[7000:8000]


In [34]:
train_data = {k: data[k] for k in train_keys}
val_data   = {k: data[k] for k in val_keys}
test_data  = {k: data[k] for k in test_keys}


In [ ]:
history_logs = []
WEIGHT_DIR = REPO_ROOT / "artifacts" / "rnn" / "weights"
RESULT_DIR = REPO_ROOT / "artifacts" / "rnn" / "results"
WEIGHT_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)

for is_lstm in [False, True]:
    arch_name = "LSTM" if is_lstm else "SimpleRNN"
    
    for var in variations:
        model_name = f"{arch_name}_{var['name']}_L{var['layers']}_H{var['hidden']}"
        print(f"\n{'='*50}\nMemulai Training: {model_name}\n{'='*50}")
        
        model = build_model(
            lstm=is_lstm, 
            layers=var['layers'], 
            hidden_state_count=var['hidden'], 
            vocab_size=VOCAB_SIZE,
            sequence_length=SEQ_LEN
        )
        
        train_gen = DataGenerator(
            mapping_data=train_data,         # <-- ganti dari data ke train_data
            image_features=image_features_map, 
            vocab_size=VOCAB_SIZE,
            sequence_length=SEQ_LEN,
            batch_size=BATCH_SIZE
        ).generate()

        val_gen = DataGenerator(             # <-- tambahkan val_gen
            mapping_data=val_data,
            image_features=image_features_map,
            vocab_size=VOCAB_SIZE,
            sequence_length=SEQ_LEN,
            batch_size=BATCH_SIZE
        ).generate()
        
        start_time = time.time()
        history = model.fit(
            train_gen, 
            steps_per_epoch=len(train_data) // BATCH_SIZE,   # <-- sesuaikan
            validation_data=val_gen,                          # <-- tambahkan
            validation_steps=len(val_data) // BATCH_SIZE,    # <-- tambahkan
            epochs=EPOCHS,
            verbose=1
        )
        end_time = time.time()
        
        model.save_weights(str(WEIGHT_DIR / f"{model_name}.h5"))
        
        duration = end_time - start_time
        history_logs.append({
            "model_type": arch_name,
            "variation_name": var['name'],
            "layers": var['layers'],
            "hidden_state": var['hidden'],
            "final_loss": history.history['loss'][-1],
            "final_val_loss": history.history['val_loss'][-1],   # <-- tambahkan
            "history_loss": history.history['loss'],             # <-- tambahkan untuk grafik
            "history_val_loss": history.history['val_loss'],     # <-- tambahkan untuk grafik
            "training_time_sec": duration
        })

df_results = pd.DataFrame(history_logs)
df_results.to_csv("hasil_variasi_model.csv", index=False)
print("\nEksperimen Selesai! Hasil disimpan di hasil_variasi_model.csv")


### Bagian 4

In [37]:
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

# Import kelas dari file teman Anda
from src.rnn.ImageCaptioningScratch import ImageCaptioningModel

eval_model = build_model(
    lstm=True, 
    layers=1, 
    hidden_state_count=128, 
    vocab_size=VOCAB_SIZE, 
    sequence_length=SEQ_LEN
)

eval_model.load_weights(str(WEIGHT_DIR / "LSTM_Shallow_Small_L1_H128.h5"))

test_image_path = "../../dataset/Images/1000268201_693b08cb0e.jpg"

model_manual = ImageCaptioningModel(
    keras_model=eval_model,
    text_util=text_util,
    is_lstm=True
)


c:\Users\HP 14s\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\layer.py:1035: UserWarning: Layer 'lambda_2' (of type Lambda) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


Model: "functional_5"

Ã¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€Â³Ã¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€Â³Ã¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€Â³Ã¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€â€œ
Ã¢â€Æ’ Layer (type)        Ã¢â€Æ’ Output Shape      Ã¢â€Æ’    Param # Ã¢â€Æ’ Connected to      Ã¢â€Æ’
Ã¢â€Â¡Ã¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€¢â€¡Ã¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€¢â€¡Ã¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€¢â€¡Ã¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€ÂÃ¢â€Â©
Ã¢â€â€š Image_Input         Ã¢â€â€š (None, 2048)      Ã¢â€â€š          0 Ã¢â€â€š -                 Ã¢â€â€š
Ã¢â€â€š (InputLayer)        Ã¢â€â€š                   Ã¢â€â€š            Ã¢â€â€š                   Ã¢â€â€š
Ã¢â€Å“Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€Â¼Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€Â¼Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€Â¼Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€Â¤
Ã¢â€â€š dense_2 (Dense)     Ã¢â€â€š (None, 256)       Ã¢â€â€š    524,544 Ã¢â€â€š Image_Input[0][0] Ã¢â€â€š
Ã¢â€Å“Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€Â¼Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€Â¼Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€Â¼Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€Â¤
Ã¢â€â€š reshape_2 (Reshape) Ã¢â€â€š (None, 1, 256)    Ã¢â€â€š          0 Ã¢â€â€š dense_2[0][0]     Ã¢â€â€š
Ã¢â€Å“Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€Â¼Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€Â¼Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€Â¼Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€Â¤
Ã¢â€â€š Caption_Input       Ã¢â€â€š (None, 35)        Ã¢â€â€š          0 Ã¢â€â€š -                 Ã¢â€â€š
Ã¢â€â€š (InputLayer)        Ã¢â€â€š                   Ã¢â€â€š            Ã¢â€â€š                   Ã¢â€â€š
Ã¢â€Å“Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€Â¼Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€

 Total params: 4,027,659 (15.36 MB)

 Trainable params: 4,027,659 (15.36 MB)

 Non-trainable params: 0 (0.00 B)

In [38]:
import os

nama_file = os.path.basename(test_image_path)

fitur_gambar = image_features_map[nama_file]
print(list(text_util.word_to_idx.keys())[:20])

hasil_kalimat = model_manual.generate_caption(image_feature=fitur_gambar, max_len=SEQ_LEN)

tokens_to_remove = ["<unk>", "<start>", "<end>"]
hasil_kalimat_bersih = ' '.join([word for word in hasil_kalimat.split() if word not in tokens_to_remove])

print("Hasil prediksi kalimat:", hasil_kalimat_bersih)


['<unk>', 'a', 'end', 'start', 'in', 'the', 'on', 'is', 'and', 'dog', 'with', 'man', 'of', 'two', 'white', 'black', 'boy', 'are', 'woman', 'girl']
Hasil prediksi kalimat: a man in a dog


In [41]:
from nltk.translate.bleu_score import sentence_bleu


hipotesis = hasil_kalimat_bersih.split()


nama_file = os.path.basename(test_image_path)  
referensi = map_image_cap.get(nama_file, []) 

referensi = [ref.split() for ref in referensi]

skor_bleu1 = sentence_bleu(referensi, hipotesis, weights=(1.0, 0, 0, 0))
print("Skor BLEU-1:", skor_bleu1)

skor_bleu4 = sentence_bleu(referensi, hipotesis, weights=(0.25, 0.25, 0.25, 0.25))
print("Skor BLEU-4:", skor_bleu4)


Skor BLEU-1: 0.26959737847033294
Skor BLEU-4: 4.1711849316776384e-155


c:\Users\HP 14s\AppData\Local\Programs\Python\Python313\Lib\site-packages\nltk\translate\bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 3-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
c:\Users\HP 14s\AppData\Local\Programs\Python\Python313\Lib\site-packages\nltk\translate\bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 4-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)


In [42]:
from nltk.translate.bleu_score import sentence_bleu

def evaluasi_model_massal(model_manual, daftar_gambar_uji, image_features_map, map_image_cap, text_util, max_len):
    total_bleu1 = 0
    total_bleu4 = 0
    jumlah_data = len(daftar_gambar_uji)
    
    tokens_to_remove = ["<unk>", "start", "end"]
    
    for nama_file in daftar_gambar_uji:
        fitur_gambar = image_features_map.get(nama_file)
        if fitur_gambar is None:
            print(f"Fitur gambar untuk {nama_file} tidak ditemukan, melewati...")
            jumlah_data -= 1
            continue
        
        hasil_kalimat = model_manual.generate_caption(image_feature=fitur_gambar, max_len=max_len)
        
        hasil_kalimat_bersih = ' '.join([word for word in hasil_kalimat.split() if word not in tokens_to_remove])
        
        referensi = map_image_cap.get(nama_file, [])
        referensi = [ref.split() for ref in referensi]
        
        if not referensi:
            print(f"Tidak ada referensi untuk {nama_file}, melewati...")
            jumlah_data -= 1
            continue
        
        skor_bleu1_gambar_ini = sentence_bleu(referensi, hasil_kalimat_bersih.split(), weights=(1.0, 0, 0, 0))
        skor_bleu4_gambar_ini = sentence_bleu(referensi, hasil_kalimat_bersih.split(), weights=(0.25, 0.25, 0.25, 0.25))
        
        total_bleu1 += skor_bleu1_gambar_ini
        total_bleu4 += skor_bleu4_gambar_ini
    
    if jumlah_data > 0:
        rata_rata_bleu1 = total_bleu1 / jumlah_data
        rata_rata_bleu4 = total_bleu4 / jumlah_data
    else:
        rata_rata_bleu1 = 0
        rata_rata_bleu4 = 0
    
    return rata_rata_bleu1, rata_rata_bleu4


In [43]:
rata_rata_bleu1, rata_rata_bleu4 = evaluasi_model_massal(
    model_manual=model_manual,
    daftar_gambar_uji=test_keys,
    image_features_map=image_features_map,
    map_image_cap=map_image_cap,
    text_util=text_util,
    max_len=SEQ_LEN
)

print(f"Rata-rata Skor BLEU-1: {rata_rata_bleu1:.4f}")
print(f"Rata-rata Skor BLEU-4: {rata_rata_bleu4:.4f}")


c:\Users\HP 14s\AppData\Local\Programs\Python\Python313\Lib\site-packages\nltk\translate\bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 2-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)


Rata-rata Skor BLEU-1: 0.3015
Rata-rata Skor BLEU-4: 0.0206


In [45]:
import numpy as np

# Smoke test scratch model yang sudah dibuat di cell evaluasi sebelumnya.
# model_manual adalah ImageCaptioningModel scratch, jadi jangan dibungkus ulang sebagai keras_model.
print("Mencoba melakukan prediksi teks dengan model scratch yang sudah dimuat...")
fitur_gambar_dummy = np.random.rand(2048)

try:
    hasil_prediksi = model_manual.generate_caption(fitur_gambar_dummy, max_len=10)
    print("Prediksi berhasil dijalankan. Hasil Teks:", hasil_prediksi)
    print("STATUS: IMPLEMENTASI SCRATCH VALID.")
except NameError:
    print("model_manual belum ada. Jalankan dulu cell yang membuat eval_model dan model_manual.")
except Exception as e:
    print("Prediksi gagal. Error saat proses propagasi maju:", e)


Mencoba melakukan prediksi teks dengan model scratch yang sudah dimuat...
Prediksi berhasil dijalankan. Hasil Teks: a man in a dog <unk>
STATUS: IMPLEMENTASI SCRATCH VALID.


## Evaluasi Lengkap RNN/LSTM
Cell berikut menjalankan evaluasi sesuai spesifikasi: seluruh variasi, Keras vs Scratch, variasi panjang caption, dan sampel kualitatif.


In [46]:
from src.rnn.experiment import (
    evaluate_all_variations,
    compare_best_keras_vs_scratch,
    max_length_sweep,
    qualitative_samples,
    write_analysis_summary,
    load_training_history,
)

# Gunakan limit kecil untuk smoke test cepat. Set EVAL_LIMIT = None untuk full test split.
EVAL_LIMIT = 100


In [47]:
variation_results, caption_details = evaluate_all_variations(REPO_ROOT, split="test", limit=EVAL_LIMIT, max_len=SEQ_LEN)
variation_results.sort_values(["model_type", "scratch_bleu_4"], ascending=[True, False])


c:\Users\HP 14s\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\layer.py:1035: UserWarning: Layer 'Drop_Image_Timestep' (of type Lambda) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


ValueError: Shape mismatch in layer #1 (named Token_Embedding)for weight Token_Embedding/embeddings. Weight expects shape (8588, 256). Received saved weight with shape (8587, 256)

In [ ]:
keras_vs_scratch = compare_best_keras_vs_scratch(REPO_ROOT, split="test", limit=EVAL_LIMIT, max_len=SEQ_LEN)
keras_vs_scratch


In [ ]:
max_length_results = max_length_sweep(REPO_ROOT, lengths=(10, 20, 35), split="test", limit=EVAL_LIMIT)
max_length_results


In [ ]:
samples = qualitative_samples(REPO_ROOT, limit=EVAL_LIMIT, n_per_bucket=4)
samples


In [ ]:
summary_path = write_analysis_summary(REPO_ROOT)
print(f"Ringkasan analisis disimpan di: {summary_path}")


## Feature Extraction Reusable
Jika `images_feature/features.npy` belum tersedia, jalankan cell ini untuk mengekstrak fitur InceptionV3 frozen dan menyimpannya ke disk.


In [ ]:
from src.rnn.feature_extraction import extract_and_save_features

# Uncomment jika perlu ekstraksi ulang fitur dari raw image.
# features_matrix, image_names = extract_and_save_features(
#     REPO_ROOT / "RNN_dataset" / "Images",
#     REPO_ROOT / "images_feature",
#     batch_size=32,
#     force=False,
# )


## Grafik Loss Training/Validation
Grafik ini dipakai untuk analisis pengaruh jumlah layer recurrent dan hidden state.


In [ ]:
import matplotlib.pyplot as plt
from src.rnn.experiment import load_training_history

history_df = load_training_history(REPO_ROOT)
for model_type, group in history_df.groupby("model_type"):
    plt.figure(figsize=(12, 5))
    for _, row in group.iterrows():
        label = f"{row['variation_name']} L{row['layers']} H{row['hidden_state']}"
        plt.plot(row["history_loss"], label=f"train {label}")
        plt.plot(row["history_val_loss"], linestyle="--", label=f"val {label}")
    plt.title(f"{model_type} Training vs Validation Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend(fontsize=8, ncol=2)
    plt.grid(True, alpha=0.3)
    plt.show()
